# 损失函数

上一章，我们构建了第一个神经网络模型，并用它预测了冰激凌的销量。但这个预测靠不靠谱，我们还不知道。

要知道预测是否准确，首先需要能够**量化预测的误差**。这正是**损失函数**（Loss Function）的作用：衡量网络模型预测值与真实值之间的差距。

损失函数是模型训练的核心。它不仅告诉我们模型"错了多少"，更是指引模型自动改进的信号（我们将在之后的两章进行探讨）。

---

最直接的量化办法就是把**预测值**和**真实值**相减，它们之间的差距称为**误差**（Error）：

$$
\text{error} = y - p
$$

这里：
* $y$：真实结果，称为**标签值**（Label）；
* $p$：网络模型的预测结果，即**预测值**。

---

直接使用误差有一个问题：当我们面对多个样本时，正的误差和负的误差会相互抵消，导致整体误差被低估。

在实践中，我们通常改用误差的平方，称为**平方误差**（Squared Error）：

$$
\text{squared error} = (y - p)^2
$$

使用平方误差有三个优点：
1. **消除正负号**：所有误差都变为正数，避免相互抵消；
2. **惩罚大误差**：误差越大，平方后惩罚越重，迫使模型优先修正大错误；
3. **处处可导**：这是下一章使用梯度下降法自动优化模型参数的数学前提。

``💡 为什么"处处可导"如此关键？因为梯度下降需要计算损失函数的导数来确定参数调整的方向。如果损失函数在某一点不可导（比如阶跃函数），梯度就无从计算，模型训练就会失败。``

## 均方误差（MSE）

当面对多个样本（例如 $n$ 天的销售数据）时，我们需要一个整体指标来衡量模型的性能。

常用的方法是计算所有样本平方误差的**平均值**，称为**均方误差**（Mean Squared Error，MSE）：

$$
\text{MSE} = \frac{1}{n} \sum\limits_{i=1}^{n} (y_i - p_i)^2
$$

这里：
* $y = [y_1, y_2, \ldots, y_n]$：$n$ 个真实结果；
* $p = [p_1, p_2, \ldots, p_n]$：$n$ 个预测结果。

MSE 越小，说明模型的整体预测越准确。MSE 为 0 意味着模型的预测与真实值完全一致，这在实践中几乎不可能，也不一定是好事（可能意味着过拟合）。

In [1]:
import numpy as np

## 数据

现在，让我们回到小明的冰激凌店。

当天晚上，销售数据出炉：一共卖出了 **165** 个冰激凌。

### 特征、标签

我们把真实结果称为**标签**，同样用 NumPy 数组来保存。

从现在起，我们的每条数据都将由两部分组成：
* **特征值**（Feature）：输入数据，即天气情况；
* **标签值**（Label）：对应的真实结果，即实际销量。


In [2]:
feature = np.array([28.1, 58.0])
label = np.array([165])

## 模型

我们继续沿用上一章搭建的模型框架（参数和推理函数）。

In [3]:
class Linear:

    def __init__(self, in_size, out_size):
        self.weight = np.ones((out_size, in_size)) / in_size
        self.bias = np.zeros(out_size)

    def __call__(self, x):
        return self.forward(x)

    def forward(self, x):
        return x @ self.weight.T + self.bias

## 损失函数（均方误差）

损失函数根据预测值 $p$ 和标签值 $y$，计算出均方误差，结果称为**损失值**（Loss）。

损失值是一个**非负数**：
* 损失值越大，说明预测误差越大，模型越不准确；
* 损失值越小，说明预测越接近真实值，模型越准确。

In [4]:
class MSELoss:

    def __call__(self, p, y):
        return self.loss(p, y)

    def loss(self, p, y):
        return np.mean(np.square(y - p))

## 建模

创建网络模型的实例的同时，我们也需要创建一个损失函数的实例。

In [5]:
layer = Linear(2, 1)
loss_fn = MSELoss()

## 推理

现在我们有了损失函数，可以正式评估模型的表现了。

首先还是进行模型推理，得到预测值。

In [6]:
prediction = layer(feature)
print(f'prediction:\t{prediction}')

prediction:	[43.05]


## 评估

有了损失函数，我们就可以量化模型推理的误差。

In [7]:
loss = loss_fn(prediction, label)
print(f'loss:\t{loss}')

loss:	14871.802500000002


损失值约为 `14872`。这个数字意味着什么？

由于我们使用的是**平方误差**，损失值的基准是销量的平方（个²）。它本身不像误差那样直观，但在计算处理时非常直接：

* 损失值越小，表示模型越准确；
* 损失值的**变化趋势**才是关键：我们的目标是让它在训练学习的过程中不断下降。

模型预测 `43` 个，真实销量是 `165` 个，误差高达 `122` 个。以这个差距来理解 `14872` 这个数字，是合理的（$122^2 \approx 14884$）。

我们的第一个模型预测严重偏低。那么，**如何让模型自动变得更准确？** 这正是下一章要开始解决的问题。

## 课后练习

1. 尝试修改权重和偏置的数值，看看能否让损失值降低。观察一下：调大权重，还是调小权重，对损失值的影响更大？

2. 当预测值与真实值的误差是 10 时，损失值是多少？误差是 100 时呢？这个规律说明了平方误差的什么特性？